**DrugGPT** ([`drug_generator.py`](https://github.com/LIYUESEN/druggpt/blob/main/drug_generator.py)):
- Build `prompt` as `p_prompt + ligand_prompt` with `p_prompt = "<|startoftext|><P>" + protein_seq + "<L>"`.
- Encode with `tokenizer.encode(prompt)`, wrap as batch `unsqueeze(0)`, move to `device`.
- `attention_mask = input_ids.ne(tokenizer.pad_token_id).float()`.
- `model.generate(..., pad_token_id=tokenizer.eos_token_id)` — first arg is **input_ids**, not the raw string.
- Decode one row and take ligand: `tokenizer.decode(...).split("<L>")[1]`.


In [ ]:
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

# Prompt layout matches drug_generator.py:
# p_prompt = "<|startoftext|><P>" + protein_seq + "<L>"; optional ligand continuation after <L>.

protein_seq = "MALWMRLLPLLALLALWGPDPAAA"
ligand_prompt = ""
p_prompt = "<|startoftext|><P>" + protein_seq + "<L>"
l_prompt = ligand_prompt
prompt = p_prompt + l_prompt
print("prompt:", prompt)


device = torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained("liyuesen/druggpt")
model = GPT2LMHeadModel.from_pretrained("liyuesen/druggpt")
model.eval()
model.to(device)

input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
attention_mask = input_ids.ne(tokenizer.pad_token_id).float()

# Defaults from drug_generator.py argparse: top_k=9, top_p=0.9, temperature=1.0, max_length=1024
with torch.inference_mode():
    sample_outputs = model.generate(
        input_ids,
        do_sample=True,
        top_k=9,
        max_length=512,
        top_p=0.9,
        temperature=1.0,
        num_return_sequences=1,
        attention_mask=attention_mask,
        pad_token_id=tokenizer.eos_token_id,
    )

decoded = tokenizer.decode(sample_outputs[0], skip_special_tokens=True)
#ligand_smiles = decoded.split("<L>", 1)[1]
#print("full decode:", decoded)
#print("ligand SMILES:", ligand_smiles)
decoded

/Users/sahilkabir/Documents/School/DrugGenGPCR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


prompt: <|startoftext|><P>MALWMRLLPLLALLALWGPDPAAA<L>


Loading weights: 100%|██████████| 149/149 [00:00<00:00, 45113.07it/s]


In [ ]:
ligand_smiles = decoded.split("<L>", 1)[1]
print("full decode:", decoded)
print("ligand SMILES:", ligand_smiles)

full decode: <P>MALWMRLLPLLALLALWGPDPAAA<L>COc1ccc(OC)c(c1)-c1cc(nc(n1)N1CCOCC1)N(C)CNENNCCKYcKYTEDILDFAKYCCFFcKYKY
ligand SMILES: COc1ccc(OC)c(c1)-c1cc(nc(n1)N1CCOCC1)N(C)CNENNCCKYcKYTEDILDFAKYCCFFcKYKY


: 

In [ ]:
import os
import numpy as np
import xgboost as xgb
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs

# Convert SMILES -> Morgan fingerprint (ECFP-style).
# Requested: ECFP6 radius 4.
ligand_smiles = "C1CCC1"
mol = Chem.MolFromSmiles(ligand_smiles)
if mol is None:
    raise ValueError(f"Invalid SMILES string: {ligand_smiles}")

fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=4, nBits=2048)
X = np.zeros((1, 2048), dtype=np.float32)
DataStructs.ConvertToNumpyArray(fp, X[0])

model_path = "xgb_rdkit_classify_fp.json"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Converted model file not found: {model_path}")

model = xgb.XGBClassifier()
model.load_model(model_path)
print(X)
pred_class = model.predict(X)[0]
print("Predicted class:", pred_class)


[23:06:30] DEPRECATION WARNING: please use MorganGenerator


[[0. 0. 1. ... 0. 0. 0.]]
Predicted class: 50
